In [18]:
from epics import PV
from time import sleep

bump definition:

Bump({'HS1P2K3RP:setCur': 0.03226,'HS3P1L4RP:setCur': 0.014116,'HS3P2L4RP:setCur': 0.014123,'HS1P1K1RP:setCur': 0.031103,'MCLKHGP:setFrq': -2.305})

In [19]:
steerers = [PV('HS1P2K3RP:setCur'), PV('HS3P1L4RP:setCur'), PV('HS3P2L4RP:setCur'), PV('HS1P1K1RP:setCur')]
steererfactors =  [0.03226, 0.014116, 0.014123, 0.031103]

In [20]:
for steerer in steerers:
    print(steerer.get())

0.0795
0.2512
-0.0614
0.0263


In [21]:
BPMs = [PV('BPMZ1K1RP:rdX'), PV('BPMZ1L2RP:rdX'), PV('BPMZ1K3RP:rdX'), PV('BPMZ1L4RP:rdX')]

In [22]:
for BPM in BPMs:
    print(BPM.get())

-2.237362
-2.6606039999999997
-1.546746
-2.4611069999999997


In [23]:
freqctrlenable = PV('MCLKHGP:ctrl:enable')
freqctrlenable.get()

0

In [24]:
gainpv = PV('AKC11VP')
enablepv = PV('AKC10VP')
refpv = PV('AKC12VP')
deadbandpv = PV('AKC13VP')

In [25]:
def bump_steerers(amount):
    for steerer, f in zip(steerers, steererfactors):
        old = steerer.get()
        steerer.put(old + amount*f)

In [26]:
#bump_steerers(0.001)

In [27]:
def get_bpm_avg():
    return sum([BPM.get() for BPM in BPMs])/4

In [28]:
get_bpm_avg()

-2.2363095

In [36]:
refpv.put(get_bpm_avg())

1

In [30]:
def get_corrector_step(verbose=True):
    gain = gainpv.get()
    ref = refpv.get()
    deadband = deadbandpv.get()
    orbit = get_bpm_avg()
    diff = ref-orbit
    if verbose:
        print(orbit)
        print(ref, gain, deadband)
        print(diff)
    if abs(diff) > deadband:
        return gain*diff
    return 0

In [31]:
get_corrector_step()

-2.2255835
-2.2559445 -0.004 0.07
-0.030361000000000082


0

In [32]:
#bump_steerers(-0.01)

In [54]:
freqctrlenable.put(0)
enablepv.put(1)
sleep(1)
while enablepv.get() != 0 and freqctrlenable.get() == 0:
    step = get_corrector_step(verbose=False)
    if step != 0:
        print('\raction:', step, '              ', end='')
        bump_steerers(step)
    sleep(0.5)
enablepv.put(0)

action: 0.00027030675000000004               

1